# ControlPlane — cascade economics

Can you tell, **before** a language model writes a single token, whether the answer
it is about to give will be wrong?

If you can, monitoring gets much cheaper. Serious checkers — LLM-as-judge, semantic
entropy, claim attribution — cost 200–1000 ms per call, so nobody runs them on all
their traffic. They sample a few percent; the rest ships unchecked.

This notebook displays what `scripts/run_all.py` measured. **It contains no logic**:
every number is read from `results/*.json`, and every table is built by a function in
`src/report.py` that is covered by the test suite. Nothing here is computed for display.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from IPython.display import Image, Markdown, display

from src.config import load_config
from src.report import (
    headline_markdown,
    latency_frame,
    load_artifacts,
    metadata_frame,
    policy_frame,
    sweep_frame,
    test_metrics_frame,
)

config = load_config(REPO_ROOT / "config.yaml")
artifacts = load_artifacts(config)
results_dir = REPO_ROOT / config.paths.results_dir
print(f"loaded {len(artifacts)} artifacts from {results_dir}")

## 1. What was run

The left-padding equivalence check is the load-bearing one. With right padding, position
`-1` of a padded batch is a pad token, every extracted activation is meaningless, and
nothing raises — the pipeline completes and returns an AUROC near 0.5 that reads as
"the idea doesn't work".

In [ ]:
metadata_frame(artifacts)

## 2. Where in the stack the signal lives

Validation AUROC for every layer and every regularisation strength tried. **The layer was
chosen here, on validation.** The test set was opened once, afterwards.

A smooth curve peaking mid-stack is itself evidence the signal is real rather than noise,
which is why the whole table is shown and not just the winner.

In [ ]:
sweep_frame(artifacts)

In [ ]:
display(Image(filename=str(results_dir / "layer_sweep.png")))

## 3. Test results — scored once

Precision and recall are reported separately and never blended into an F1. The two failure
modes differ in cost by orders of magnitude: a false positive wastes one judge call, a
false negative lets a user act on a wrong answer. The probe is tuned for recall and low
precision is accepted by design.

In [ ]:
test_metrics_frame(artifacts)

In [ ]:
display(Image(filename=str(results_dir / "roc_curve.png")))

## 4. The three policies

All at N = 1,000,000 responses. Rows two and three spend **the same judge budget**.

Coverage and verdict are different things. Every response is scored by the probe; only the
expensive verdict is rationed. Random sampling has a few percent coverage *and* a few
percent verdict. That gap is the whole result.

In [ ]:
policy_frame(artifacts)

## 5. The headline

In [ ]:
display(Markdown(headline_markdown(artifacts)))

## 6. Does it slow the model down?

No, and this is measured rather than claimed. The probe adds **no additional forward
pass**: the activation it reads is a by-product of the prefill that generation already
performs, so its marginal cost is one scale-and-dot-product.

In [ ]:
latency_frame(artifacts)

## 7. What this does not show

The full limitations section is in [`results/RESULTS.md`](../results/RESULTS.md), written
from the artifacts rather than from boilerplate. The short version: one model, one dataset,
knowledge questions only, a single seed, and automatic alias matching standing in for human
judgment. This measures the probe, not an end-to-end system.

In [ ]:
display(Markdown((results_dir / "RESULTS.md").read_text(encoding="utf-8")))